In [1]:
# Source code generated by Vnpt ETL - Data team - eGov
# Date: 2024-11-29 01:23:02
# Additional dependencies: 
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

      
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
os.environ["AWS_REGION"] = "us-east-1"
import pyspark.pandas as ps
ps.options.compute.ops_on_diff_frames = True
from datetime import datetime, timedelta

In [2]:
try:
    SparkSession1 = SparkSession.builder \
        .appName("test_pipeline_danhsachcongviechoso_donvi") \
        .getOrCreate()
    print("SparkSession created successfully")
except Exception as e:
    print("Error creating SparkSession:", e)
# SparkSession1 = SparkSession.builder \
#     .master("spark://spark-master-service:7077") \
#     .appName("dvc_dm_coquan") \
#     .config("spark.cores.max", "2") \
#     .config("spark.executor.memory", "2g") \
#     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
#     .config("spark.sql.catalog.khodldvc", "org.apache.iceberg.spark.SparkCatalog") \
#     .config("spark.sql.catalog.khodldvc.type", "hive") \
#     .config("spark.sql.catalog.khodldvc.uri", "thrift://hive-metastore.lakehouse-saq1:9083") \
#     .config("spark.sql.catalog.khodldvc.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
#     .config("spark.sql.catalog.khodldvc.warehouse", "s3a://lakehouse/lakehouse_ctl") \
#     .config("spark.sql.catalog.khodldvc.s3.endpoint", "http://10.168.6.104:9000") \
#     .config("spark.sql.catalog.khodldvc.s3.access-key-id", "minioadmin") \
#     .config("spark.sql.catalog.khodldvc.s3.secret-access-key", "minioadmin") \
#     .config("spark.sql.defaultCatalog", "khodldvc") \
#     .getOrCreate()

try:
    SparkLoadTable1 = SparkSession1.read.format("iceberg").load('bdh_mdm.dvc_md_danhsachcongviechoso_otm_test')
    SparkLoadTable1 = SparkLoadTable1.pandas_api()
except Exception as e:
    print("Error load table:", e)

24/11/29 17:02:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Error load table: [TABLE_OR_VIEW_NOT_FOUND] The table or view bdh_mdm.dvc_md_danhsachcongviechoso_otm_test cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.


In [ ]:
formulaRow1 = SparkLoadTable1.copy()
formulaRow1['ngaygiotiepnhan'] = formulaRow1.apply(lambda x: x["ngaygiotiepnhan"]- pd.Timedelta(hours=7), axis=1)
formulaRow1['ngaygiobdcvhs'] = formulaRow1.apply(lambda x: x["ngaygiobdcvhs"]- pd.Timedelta(hours=7), axis=1)
formulaRow1['ngaygiobdcvhsonline'] = formulaRow1.apply(lambda x: x["ngaygiobdcvhsonline"]- pd.Timedelta(hours=7), axis=1)
formulaRow1['ngaygioktcvhs'] = formulaRow1.apply(lambda x: x["ngaygioktcvhs"]- pd.Timedelta(hours=7), axis=1)
formulaRow1['ngaygioktcvhsonline'] = formulaRow1.apply(lambda x: x["ngaygioktcvhsonline"]- pd.Timedelta(hours=7), axis=1)
formulaRow1['ngaygioktcvhsdh'] = formulaRow1.apply(lambda x: x["ngaygioktcvhsdh"]- pd.Timedelta(hours=7), axis=1)

In [ ]:
formulaRow2 = formulaRow1.copy()
formulaRow2['ngaybdcvhs'] = formulaRow2.apply(lambda x: str(x['ngaygiobdcvhs']).split(' ')[0], axis=1)

In [ ]:
formulaRow3 = formulaRow2.copy()
formulaRow3['dunghan'] = formulaRow3.apply(lambda x: 1 if x['ngaygioktcvhsdh'] is not None and x['ngaygioktcvhs'] is not None and x['ngaygioktcvhsdh'] >= x['ngaygioktcvhs']
    else 0 if x['ngaygioktcvhsdh'] is not None and x['ngaygioktcvhs'] is not None
    else None, axis=1)
formulaRow3['onlinedunghan'] = formulaRow3.apply(lambda x: 1 if x['ngaygioktcvhsdh'] is not None and x['ngaygioktcvhsonline'] is not None and x['ngaygioktcvhsdh'] >= x['ngaygioktcvhsonline']
    else 0 if x['ngaygioktcvhsdh'] is not None and x['ngaygioktcvhsonline'] is not None
    else None, axis=1)

In [ ]:
formulaRow4 = formulaRow3.copy()
formulaRow4['tgluugiucvhs'] = formulaRow4.apply(lambda x: ((x['ngaygioktcvhs'].timestamp() - x['ngaygiotiepnhan'].timestamp()) / 86400)
    if pd.notna(x['ngaygioktcvhs']) and pd.notna(x['ngaygiotiepnhan']) else None, axis=1)
formulaRow4['tgchobatdaucvhs'] = formulaRow4.apply(lambda x: ((x['ngaygiobdcvhs'].timestamp() - x['ngaygiotiepnhan'].timestamp()) / 86400)
    if pd.notna(x['ngaygiobdcvhs']) and pd.notna(x['ngaygiotiepnhan']) else None, axis=1)
formulaRow4['tgxulycvhs'] = formulaRow4.apply(lambda x: ((x['ngaygioktcvhs'].timestamp() - x['ngaygiobdcvhs'].timestamp()) / 86400)
    if pd.notna(x['ngaygioktcvhs']) and pd.notna(x['ngaygiobdcvhs']) else None, axis=1)
formulaRow4['tgxulycvhsonline'] = formulaRow4.apply(lambda x: ((x['ngaygioktcvhsonline'].timestamp() - x['ngaygiobdcvhsonline'].timestamp()) / 86400)
    if pd.notna(x['ngaygioktcvhsonline']) and pd.notna(x['ngaygiobdcvhsonline']) else None, axis=1)

In [ ]:
customTransformations1 = formulaRow4
customTransformations1['tgluugiucvhs'] = customTransformations1['tgluugiucvhs'].round(5)
customTransformations1['tgchobatdaucvhs'] = customTransformations1['tgchobatdaucvhs'].round(5)
customTransformations1['tgxulycvhs'] = customTransformations1['tgxulycvhs'].round(5)
customTransformations1['tgxulycvhsonline'] = customTransformations1['tgxulycvhsonline'].round(5)

In [ ]:
formulaRow5 = customTransformations1.copy()
formulaRow5['slcvhsdunghan'] = formulaRow5.apply(lambda x: 1 if x['dunghan'] == 1 else 0, axis=1)
formulaRow5['slcvhsquahan'] = formulaRow5.apply(lambda x: 1 if x['dunghan'] == 0 else 0, axis=1)
formulaRow5['slcvhsonlinedunghan'] = formulaRow5.apply(lambda x: 1 if x['onlinedunghan'] == 1 else 0, axis=1)
formulaRow5['slcvhsonlinequahan'] = formulaRow5.apply(lambda x: 1 if x['onlinedunghan'] == 0 else 0, axis=1)
formulaRow5['cvhskhac'] = formulaRow5.apply(lambda x: 1 if pd.isna(x['dunghan']) else 0, axis=1)
formulaRow5['tgtbxulycvhsdunghan'] = formulaRow5.apply(lambda x: x['tgxulycvhs'] if x['dunghan'] == 1 else None, axis=1)
formulaRow5['tgtbxulycvhsquahan'] = formulaRow5.apply(lambda x: x['tgxulycvhs'] if x['dunghan'] == 0 else None, axis=1)
formulaRow5['tgtbxulycvhsonlinedunghan'] = formulaRow5.apply(lambda x: x['tgxulycvhsonline'] if x['onlinedunghan'] == 1 else None, axis=1)
formulaRow5['tgtbxulycvhsonlinequahan'] = formulaRow5.apply(lambda x: x['tgxulycvhsonline'] if x['onlinedunghan'] == 0 else None, axis=1)

In [ ]:
aggregate1 = formulaRow5.groupby(["maloaicongviec","trangthaicongviechoso","matochuyenmonthuchien","macanbothuchien","malinhvuc","ngaybdcvhs"]).agg(slcvhsdunghan_sum=('slcvhsdunghan', 'sum'), slcvhsquahan_sum=('slcvhsquahan', 'sum'), slcvhsonlinedunghan_sum=('slcvhsonlinedunghan', 'sum'), slcvhsonlinequahan_sum=('slcvhsonlinequahan', 'sum'), cvhskhac_sum=('cvhskhac', 'sum'), tgtbxulycvhsdunghan_sum=('tgtbxulycvhsdunghan', 'sum'), tgtbxulycvhsquahan_sum=('tgtbxulycvhsquahan', 'sum'), tgtbxulycvhsonlinedunghan_sum=('tgtbxulycvhsonlinedunghan', 'sum'), tgtbxulycvhsonlinequahan_sum=('tgtbxulycvhsonlinequahan', 'sum'), tgluugiucvhs_sum=('tgluugiucvhs', 'sum'), tgchobatdaucvhs_sum=('tgchobatdaucvhs', 'sum'), tgxulycvhs_sum=('tgxulycvhs', 'sum'), tgxulycvhsonline_sum=('tgxulycvhsonline', 'sum')).reset_index()

In [ ]:
# Rename columns
rename1 = aggregate1.rename(columns={"slcvhsdunghan_sum": "slcvhsdunghan", "slcvhsquahan_sum": "slcvhsquahan", "slcvhsonlinedunghan_sum": "slcvhsonlinedunghan", "slcvhsonlinequahan_sum": "slcvhsonlinequahan", "cvhskhac_sum": "cvhskhac", "tgtbxulycvhsdunghan_sum": "tgtbxulycvhsdunghan", "tgtbxulycvhsquahan_sum": "tgtbxulycvhsquahan", "tgtbxulycvhsonlinedunghan_sum": "tgtbxulycvhsonlinedunghan", "tgtbxulycvhsonlinequahan_sum": "tgtbxulycvhsonlinequahan", "tgluugiucvhs_sum": "tgluugiucvhs", "tgchobatdaucvhs_sum": "tgchobatdaucvhs", "tgxulycvhs_sum": "tgxulycvhs", "tgxulycvhsonline_sum": "tgxulycvhsonline"})

In [ ]:
# Initialize the output DataFrame
typeConverter1 = rename1.copy()
# Convert ngaybdcvhs from object to datetime64
typeConverter1["ngaybdcvhs"] = ps.to_datetime(rename1["ngaybdcvhs"], format="mixed")

In [ ]:
customTransformations2 = typeConverter1
customTransformations2['tgtbxulycvhsdunghan'] = (customTransformations2['tgtbxulycvhsdunghan'] / customTransformations2['slcvhsdunghan']).round(2)
customTransformations2['tgtbxulycvhsquahan'] = (customTransformations2['tgtbxulycvhsquahan'].abs() / customTransformations2['slcvhsquahan']).round(2)
customTransformations2['tgtbxulycvhsonlinedunghan'] = (customTransformations2['tgtbxulycvhsonlinedunghan'] / customTransformations2['slcvhsonlinedunghan']).round(2)
customTransformations2['tgtbxulycvhsonlinequahan'] = (customTransformations2['tgtbxulycvhsonlinequahan'].abs() / customTransformations2['slcvhsonlinequahan']).round(2)
customTransformations2['tgluugiucvhs'] = customTransformations2['tgluugiucvhs'].round(2)
customTransformations2['tgchobatdaucvhs'] = customTransformations2['tgchobatdaucvhs'].round(2)
customTransformations2['tgxulycvhs'] = customTransformations2['tgxulycvhs'].round(2)
customTransformations2['tgxulycvhsonline'] = customTransformations2['tgxulycvhsonline'].round(2)

In [ ]:
formulaRow6 = customTransformations2.copy()
formulaRow6['ngaycapnhat'] = formulaRow6.apply(lambda x: ps.to_datetime('now'), axis=1)
formulaRow6['maky'] = formulaRow6.apply(lambda x: x['ngaycapnhat'].strftime('%Y%m'), axis=1)

In [ ]:
formulaRow6 = formulaRow6.to_spark()
formulaRow6.write.format("iceberg").mode("overwrite").saveAsTable('bdh_mdm.fact_dvc_danhsachcongviechoso_donvi_test')

In [ ]:
SparkSession1.stop()